# Benchmarks: the final model from its checkpoints

**ML4CV project, University of Bologna. Christian Joseph Fiore.**

This notebook re-evaluates the final model from its saved weights (nothing is trained) and rebuilds every benchmark table: SwinMTL Tab. I, II, III and V, DINOv2 (Cityscapes segmentation, NYUv2 depth), DPT (NYUv2 depth), zero-shot transfer and the target scorecard. Requirements and run time are in the README.

**Final model**, 3 seeds (42, 43, 44) per dataset: DINOv2 ViT-S/14 + DPT, 8 fine-tuned ViT blocks, no critic, linear depth output, patch mixing. Cityscapes with frame-consistent position embeddings (`ckpt/final/cityscapes_base_s{seed}`); NYUv2 with standard position embeddings (`ckpt/seeds/nc8_s{seed}`).

Headline numbers are single pass; test-time augmentation (TTA) rows are supplementary. Results go to `results/reproduce/benchmarks/`, cached (`FORCE = True` recomputes).

In [1]:
FORCE = False       # recompute the evaluations even if cached in results/reproduce/benchmarks
TTA = True          # also multi-scale (CS seg), flip and sliding window (supplementary rows)
ZERO_SHOT = True    # KITTI, SUN RGB-D, ADE20K / VOC / Pascal Context subsets

## Setup

***NOTE:*** The DINOv2 and DPT code is used straight from the local repositories `./dinov2` and `./DPT`.

In [2]:
import os, sys
if os.path.basename(os.getcwd()) == "experiments":
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

In [3]:
from repro_common import *

setup()
OUT = REPRO_DIR / "benchmarks"
OUT.mkdir(parents=True, exist_ok=True)
print(f"Device: {device} ({torch.cuda.get_device_name(0) if device.type == 'cuda' else 'no GPU'}), torch {torch.__version__}")
print(f"Checkpoints: {ROOT / 'ckpt'}, data: {CFG['data_root']}, output: {OUT}")

Device: cuda (NVIDIA RTX A4000), torch 2.11.0+cu128
Checkpoints: /home/chrisjf/Documents/ml4cv/ckpt, data: /home/chrisjf/Documents/ml4cv/data, output: /home/chrisjf/Documents/ml4cv/results/reproduce/benchmarks


## The final model
One entry per seed: the checkpoint folder under `ckpt/` and the settings that matter at test time.
- `frame_pe` changes which position embeddings a sliding window receives.
- The variant (`final`, `partial_ft8`, i.e. 8 fine-tuned blocks) is read from the folder name.

In [4]:
SEEDS = [42, 43, 44]
FINAL = {
    "cityscapes": {s: (f"final/cityscapes_base_s{s}/cityscapes_final", dict(frame_pe=True)) for s in SEEDS},
    "nyu": {s: (f"seeds/nc8_s{s}/nyu_partial_ft8", dict(frame_pe=False)) for s in SEEDS},
}
for dataset, runs in FINAL.items():
    for seed, (run, overrides) in runs.items():
        path = ROOT / "ckpt" / run / "final.pt"
        print(f"{dataset} seed {seed}: {path.relative_to(ROOT)}  {'ok' if path.exists() else 'MISSING'}  {overrides}")
        assert path.exists(), f"missing checkpoint {path}"

cityscapes seed 42: ckpt/final/cityscapes_base_s42/cityscapes_final/final.pt  ok  {'frame_pe': True}
cityscapes seed 43: ckpt/final/cityscapes_base_s43/cityscapes_final/final.pt  ok  {'frame_pe': True}
cityscapes seed 44: ckpt/final/cityscapes_base_s44/cityscapes_final/final.pt  ok  {'frame_pe': True}
nyu seed 42: ckpt/seeds/nc8_s42/nyu_partial_ft8/final.pt  ok  {'frame_pe': False}
nyu seed 43: ckpt/seeds/nc8_s43/nyu_partial_ft8/final.pt  ok  {'frame_pe': False}
nyu seed 44: ckpt/seeds/nc8_s44/nyu_partial_ft8/final.pt  ok  {'frame_pe': False}


## Evaluation
For each seed:
- **Cityscapes:**
  - depth at 512×1024 (SwinMTL Tab. I) and at full resolution (Tab. II);
  - segmentation, 19 classes, standard and SwinMTL per-image mIoU;
  - with TTA: multi-scale + flip segmentation and the sliding window.
- **NYUv2:** depth on the 654 test images with the Eigen crop, 40-class segmentation; with TTA, flip and the sliding window.
- **Zero-shot:**
  - Cityscapes model → KITTI Eigen;
  - NYUv2 model → SUN RGB-D;
  - both → ADE20K, VOC 2012 and Pascal Context, on mapped class subsets.

In [5]:
EVAL, ZS = {}, {}
for dataset, runs in FINAL.items():
    for seed, (run, overrides) in runs.items():
        def compute():
            model = load_run(run, dataset, **overrides)
            if dataset == "cityscapes":
                r = eval_cityscapes(model, ms=TTA, slide=TTA)
            else:
                r = eval_nyu(model, slide=TTA)
            del model
            torch.cuda.empty_cache()
            return r
        EVAL[(dataset, seed)] = cached(OUT / f"{dataset}_s{seed}{'' if TTA else '_notta'}.json", compute, FORCE)

        if ZERO_SHOT:
            def compute_zs():
                model = load_run(run, dataset, **overrides)
                r = zero_shot(model, dataset)
                del model
                torch.cuda.empty_cache()
                return r
            ZS[(dataset, seed)] = cached(OUT / f"{dataset}_s{seed}_zeroshot.json", compute_zs, FORCE)

        r = EVAL[(dataset, seed)]
        if dataset == "cityscapes":
            print(f"CS  s{seed}: AbsRel {r['cs_depth_512x1024']['abs_rel']:.4f}, RMSE {r['cs_depth_512x1024']['rmse']:.3f}, mIoU {r['cs_seg']['mIoU']:.2f}")
        else:
            print(f"NYU s{seed}: RMSE {r['nyu_depth']['rmse']:.4f}, AbsRel {r['nyu_depth']['abs_rel']:.4f}, mIoU {r['nyu_seg']['mIoU']:.2f} "
                  f"(per-image {r['nyu_seg_swinmtl']['mIoU']:.2f})")

CS  s42: AbsRel 0.1059, RMSE 5.046, mIoU 77.95
CS  s43: AbsRel 0.1100, RMSE 5.018, mIoU 77.59
CS  s44: AbsRel 0.1089, RMSE 5.010, mIoU 77.67
NYU s42: RMSE 0.4205, AbsRel 0.1215, mIoU 58.25 (per-image 58.55)
NYU s43: RMSE 0.4205, AbsRel 0.1198, mIoU 58.16 (per-image 58.56)
NYU s44: RMSE 0.4183, AbsRel 0.1199, mIoU 57.88 (per-image 58.18)


Model size and speed (SwinMTL Tab. V). GMACs at 512×1024 with `FlopCounterMode`, attention included; FPS in bf16 with batch 1 (it depends on the GPU). SwinMTL's 65 G FLOPs has no stated input size, so only the parameter counts are comparable.

In [6]:
setup(frame_pe=True)
model = build_model("cityscapes", "final", pretrained=False).to(device)
EFF = efficiency(model, CFG["datasets"]["cityscapes"]["base_size"])
del model
torch.cuda.empty_cache()
save_json(EFF, OUT / "efficiency.json")
print({k: EFF[k] for k in ("params_M", "trainable_M", "GMACs", "fps") if k in EFF})

{'params_M': 37.806665, 'trainable_M': 29.952713, 'GMACs': 475.667838976, 'fps': 24.593985323728763}


## Benchmark tables
Final model = mean ± std over the 3 seeds. Paper rows are single runs as reported. Rows marked *TTA* are supplementary. mIoU: papers as reported, mine with the standard metric, SwinMTL's per-image metric in its own column.

In [7]:
def vals(dataset, key, metric, source=None):
    source = EVAL if source is None else source
    return np.array([source[(dataset, s)][key][metric] for s in SEEDS if key in source.get((dataset, s), {})], dtype=float)


def pm(dataset, key, metric, source=None):
    v = vals(dataset, key, metric, source)
    return None if len(v) == 0 else f"{v.mean():.4g} ± {v.std(ddof=1):.2g}"


def table(rows, name):
    df = pd.DataFrame(rows).set_index("model")
    df.to_csv(OUT / f"{name}.csv")
    display(df.fillna("—"))
    return df


OURS = "Ours (final, 3 seeds)"
D_KEYS = ["abs_rel", "sq_rel", "rmse", "rmse_log", "d1", "d2", "d3"]

print("SwinMTL Tab. I: Cityscapes depth at 512x1024 (* = self-supervised in the paper)")
rows = [dict(model=m, source="paper", **v) for m, v in PAPER["swinmtl_tab1"].items()]
rows.append(dict(model=OURS, source="measured", res="512x1024", **{m: pm("cityscapes", "cs_depth_512x1024", m) for m in D_KEYS}))
if TTA:
    rows.append(dict(model=OURS + " + sliding window (TTA)", source="measured, TTA", res="512x1024",
                     **{m: pm("cityscapes", "cs_depth_512x1024_slide", m) for m in D_KEYS}))
tab1 = table(rows, "swinmtl_tab1")

print("SwinMTL Tab. II: Cityscapes multi-task at 1024x2048")
rows = [{"model": m, "source": "paper", "RMSE": r, "mIoU (as reported)": i} for m, (r, i) in PAPER["swinmtl_tab2"].items()]
rows.append({"model": OURS, "source": "measured", "RMSE": pm("cityscapes", "cs_depth_full", "rmse"),
             "mIoU (standard, 19 cl.)": pm("cityscapes", "cs_seg", "mIoU"), "mIoU (SwinMTL per-image)": pm("cityscapes", "cs_seg_swinmtl", "mIoU")})
if TTA:
    rows.append({"model": OURS + " + sliding window (TTA)", "source": "measured, TTA", "RMSE": pm("cityscapes", "cs_depth_full_slide", "rmse"),
                 "mIoU (standard, 19 cl.)": pm("cityscapes", "cs_seg_slide", "mIoU"), "mIoU (SwinMTL per-image)": pm("cityscapes", "cs_seg_swinmtl_slide", "mIoU")})
tab2 = table(rows, "swinmtl_tab2")

print("SwinMTL Tab. III: NYUv2 multi-task (40 classes)")
rows = [{"model": m, "source": "paper", "RMSE": r, "mIoU (as reported)": i} for m, (r, i) in PAPER["swinmtl_tab3"].items()]
rows.append({"model": OURS, "source": "measured", "RMSE": pm("nyu", "nyu_depth", "rmse"),
             "mIoU (standard)": pm("nyu", "nyu_seg", "mIoU"), "mIoU (SwinMTL per-image)": pm("nyu", "nyu_seg_swinmtl", "mIoU")})
if TTA:
    rows.append({"model": OURS + " + sliding window (TTA)", "source": "measured, TTA", "RMSE": pm("nyu", "nyu_depth_slide", "rmse"),
                 "mIoU (standard)": pm("nyu", "nyu_seg_slide", "mIoU"), "mIoU (SwinMTL per-image)": pm("nyu", "nyu_seg_swinmtl_slide", "mIoU")})
tab3 = table(rows, "swinmtl_tab3")

print("SwinMTL Tab. V: model size (ours: GMACs at 512x1024; SwinMTL does not state the input size of its FLOPs, so only the parameter counts are comparable)")
rows = [{"model": m, "source": "paper", "FLOPs (G)": f, "params (M)": p} for m, (f, p) in PAPER["swinmtl_tab5"].items()]
rows.append({"model": "Ours (8 blocks)", "source": f"measured @ {EFF['input']}", "FLOPs (G)": EFF["GMACs"], "params (M)": EFF["params_M"],
             "trainable (M)": EFF.get("trainable_M"), "FPS": EFF.get("fps")})
tab5 = table(rows, "swinmtl_tab5")

SwinMTL Tab. I: Cityscapes depth at 512x1024 (* = self-supervised in the paper)


,source,res,abs_rel,sq_rel,rmse,rmse_log,d1,d2,d3
model,,,,,,,,,
GLNet,paper,1024x2048,0.111,—,6.437,0.182,0.868,0.961,0.983
Pilzer et al.*,paper,512x256,0.438,5.713,5.745,0.4,0.711,0.877,0.94
Struct2Depth*,paper,416x128,0.145,1.737,7.28,0.205,0.813,0.942,0.976
Li et al.*,paper,416x128,0.119,1.29,6.98,0.19,0.846,0.952,0.982
DynamicDepth*,paper,416x128,0.103,1.0,5.867,0.157,0.895,0.974,0.991
Lee et al.*,paper,832x256,0.116,1.213,6.695,0.186,0.852,0.951,0.982
InstaDM*,paper,832x256,0.111,1.158,6.437,0.182,0.868,0.961,0.983
SwinMTL,paper,512x1024,0.089,1.051,5.481,0.139,0.921,0.976,0.99
"Ours (final, 3 seeds)",measured,512x1024,0.1082 ± 0.0021,0.8228 ± 0.0027,5.024 ± 0.019,0.1461 ± 0.00094,0.914 ± 0.00034,0.9815 ± 7.6e-05,0.9939 ± 6.9e-05


SwinMTL Tab. II: Cityscapes multi-task at 1024x2048


/tmp/ipykernel_1069987/1873729846.py:14: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  display(df.fillna("—"))


,source,RMSE,mIoU (as reported),"mIoU (standard, 19 cl.)",mIoU (SwinMTL per-image)
model,,,,,
Hybridnet,paper,12.09,66.61,—,—
MGMNet,paper,8.3,55.7,—,—
MTL,paper,6.797,70.43,—,—
PAD-Net,paper,6.777,70.23,—,—
3-ways,paper,6.528,75.0,—,—
[44],paper,6.649,74.95,—,—
SwinMTL,paper,6.352,76.41,—,—
"Ours (final, 3 seeds)",measured,5.059 ± 0.019,—,77.74 ± 0.19,65.23 ± 0.017
"Ours (final, 3 seeds) + sliding window (TTA)","measured, TTA",4.69 ± 0.008,—,78.98 ± 0.19,67.44 ± 0.064


SwinMTL Tab. III: NYUv2 multi-task (40 classes)


/tmp/ipykernel_1069987/1873729846.py:14: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  display(df.fillna("—"))


,source,RMSE,mIoU (as reported),mIoU (standard),mIoU (SwinMTL per-image)
model,,,,,
Cross-Stitch,paper,0.629,36.34,—,—
MTL,paper,0.638,39.44,—,—
PAD-Net,paper,0.627,36.61,—,—
3-ways,paper,0.622,39.47,—,—
[44],paper,0.604,38.93,—,—
MTAN,paper,0.5906,40.01,—,—
MTI-Net,paper,0.5365,45.97,—,—
ATRC,paper,0.5363,46.33,—,—
InvPT,paper,0.5183,53.56,—,—


SwinMTL Tab. V: model size (ours: GMACs at 512x1024; SwinMTL does not state the input size of its FLOPs, so only the parameter counts are comparable)


/tmp/ipykernel_1069987/1873729846.py:14: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  display(df.fillna("—"))


,source,FLOPs (G),params (M),trainable (M),FPS
model,,,,,
TaskPrompter,paper,416.000000,373.000000,—,—
ATRC w/ ViT-B,paper,260.000000,119.000000,—,—
PAD-Net w/ ViT-B,paper,166.000000,109.000000,—,—
SwinMTL w/ Swin-B,paper,65.000000,87.380000,—,—
Ours (8 blocks),measured @ 512x1024,475.667839,37.806665,29.952713,24.593985


In [8]:
dd, dseg = PAPER["dinov2_tab11_vits14"], PAPER["dinov2_tab10_vits14"]

print("DINOv2 Tab. 10: Cityscapes segmentation, mIoU (19 classes)")
rows = [{"model": "DINOv2 ViT-S/14 (frozen)", "source": "paper", "single scale (lin.)": dseg["cityscapes"][0], "multi-scale": dseg["cityscapes"][1]}]
rows.append({"model": OURS, "source": "measured", "single scale (lin.)": pm("cityscapes", "cs_seg", "mIoU"),
             "multi-scale": pm("cityscapes", "cs_seg_ms", "mIoU") if TTA else None})
table(rows, "dinov2_cs_seg")

print("DINOv2 Tab. 11: NYUv2 depth, RMSE (DINOv2 trained on ~24k frames, ours on the 795 labelled images)")
rows = [{"model": f"DINOv2 ViT-S/14 {h}", "source": "paper", "RMSE": dd["nyu"][i]} for i, h in enumerate(["lin. 1", "lin. 4", "DPT"])]
rows.append({"model": OURS, "source": "measured", "RMSE": pm("nyu", "nyu_depth", "rmse"),
             "RMSE flip (TTA)": pm("nyu", "nyu_depth_flip", "rmse"), "RMSE sliding (TTA)": pm("nyu", "nyu_depth_slide", "rmse") if TTA else None})
table(rows, "dinov2_nyu_depth")

print("DPT Tab. 2: NYUv2 depth")
K = ["d1", "d2", "d3", "abs_rel", "rmse", "log10"]
rows = [dict(model=m, source="paper (~24k frames)", **v) for m, v in PAPER["dpt_tab2"].items()]
rows.append(dict(model=OURS, source="measured (795 images)", **{m: pm("nyu", "nyu_depth", m) for m in K}))
if TTA:
    rows.append(dict(model=OURS + " + sliding window (TTA)", source="measured, TTA", **{m: pm("nyu", "nyu_depth_slide", m) for m in K}))
table(rows, "dpt_nyu")

DINOv2 Tab. 10: Cityscapes segmentation, mIoU (19 classes)


,source,single scale (lin.),multi-scale
model,,,
DINOv2 ViT-S/14 (frozen),paper,66.6,77.1
"Ours (final, 3 seeds)",measured,77.74 ± 0.19,79.79 ± 0.13


DINOv2 Tab. 11: NYUv2 depth, RMSE (DINOv2 trained on ~24k frames, ours on the 795 labelled images)


,source,RMSE,RMSE flip (TTA),RMSE sliding (TTA)
model,,,,
DINOv2 ViT-S/14 lin. 1,paper,0.449,—,—
DINOv2 ViT-S/14 lin. 4,paper,0.417,—,—
DINOv2 ViT-S/14 DPT,paper,0.356,—,—
"Ours (final, 3 seeds)",measured,0.4197 ± 0.0013,0.4136 ± 0.0013,0.4096 ± 0.00084


DPT Tab. 2: NYUv2 depth


,source,d1,d2,d3,abs_rel,rmse,log10
model,,,,,,,
DORN,paper (~24k frames),0.828,0.965,0.992,0.115,0.509,0.051
VNL,paper (~24k frames),0.875,0.976,0.994,0.111,0.416,0.048
BTS,paper (~24k frames),0.885,0.978,0.994,0.11,0.392,0.047
DPT-Hybrid,paper (~24k frames),0.904,0.988,0.998,0.11,0.357,0.045
"Ours (final, 3 seeds)",measured (795 images),0.8652 ± 0.0014,0.9833 ± 0.00068,0.9971 ± 2e-05,0.1204 ± 0.00096,0.4197 ± 0.0013,0.0505 ± 0.00033
"Ours (final, 3 seeds) + sliding window (TTA)","measured, TTA",0.8736 ± 0.00056,0.9848 ± 0.00058,0.9973 ± 8.7e-05,0.1159 ± 0.00057,0.4096 ± 0.00084,0.04905 ± 0.00022


,source,d1,d2,d3,abs_rel,rmse,log10
model,,,,,,,
DORN,paper (~24k frames),0.828,0.965,0.992,0.115,0.509,0.051
VNL,paper (~24k frames),0.875,0.976,0.994,0.111,0.416,0.048
BTS,paper (~24k frames),0.885,0.978,0.994,0.11,0.392,0.047
DPT-Hybrid,paper (~24k frames),0.904,0.988,0.998,0.11,0.357,0.045
"Ours (final, 3 seeds)",measured (795 images),0.8652 ± 0.0014,0.9833 ± 0.00068,0.9971 ± 2e-05,0.1204 ± 0.00096,0.4197 ± 0.0013,0.0505 ± 0.00033
"Ours (final, 3 seeds) + sliding window (TTA)","measured, TTA",0.8736 ± 0.00056,0.9848 ± 0.00058,0.9973 ± 8.7e-05,0.1159 ± 0.00057,0.4096 ± 0.00084,0.04905 ± 0.00022


### Zero-shot transfer (reference only, not comparable with in-domain numbers)
- KITTI and SUN RGB-D images are resized so that their focal length matches the training camera.
- *Median-scaled* rescales each prediction by its median ratio to the ground truth, which removes the global scale and measures structure only.

In [9]:
if ZERO_SHOT:
    print("KITTI Eigen (652 images), Cityscapes model")
    K = ["d1", "abs_rel", "rmse", "rmse_log"]
    rows = [dict(model=m, source="paper, trained on KITTI (DPT Tab. 3)", **{k: v[k] for k in K}) for m, v in PAPER["dpt_tab3"].items()]
    for key, tag in [("kitti_raw", "raw"), ("kitti_median", "median-scaled")]:
        rows.append(dict(model=f"{OURS}, zero-shot {tag}", source="measured", **{m: pm("cityscapes", key, m, ZS) for m in K}))
    table(rows, "zeroshot_kitti")

    print("SUN RGB-D test, NYUv2 model")
    rows = [dict(model=f"DINOv2 ViT-S/14 {h}", source="paper, trained on ~24k NYU frames", rmse=r) for h, r in zip(["lin. 1", "lin. 4", "DPT"], dd["sun"])]
    for key, tag in [("sun_raw", "raw"), ("sun_median", "median-scaled")]:
        rows.append(dict(model=f"{OURS}, {tag}", source="measured", **{m: pm("nyu", key, m, ZS) for m in K}))
    table(rows, "zeroshot_sun")

    print("Segmentation on the mapped class subsets")
    rows = []
    for dataset in ("cityscapes", "nyu"):
        for target in ("ade20k", "voc2012", "pcontext"):
            key = f"{target}_subset"
            rows.append(dict(model=f"{dataset} -> {target}", classes=ZS[(dataset, SEEDS[0])][key]["n_classes"],
                             mIoU=pm(dataset, key, "mIoU", ZS), pixAcc=pm(dataset, key, "pixAcc", ZS)))
    table(rows, "zeroshot_seg")

KITTI Eigen (652 images), Cityscapes model


,source,d1,abs_rel,rmse,rmse_log
model,,,,,
DORN,"paper, trained on KITTI (DPT Tab. 3)",0.932,0.072,2.626,0.12
VNL,"paper, trained on KITTI (DPT Tab. 3)",0.938,0.072,3.258,0.117
BTS,"paper, trained on KITTI (DPT Tab. 3)",0.956,0.059,2.756,0.096
DPT-Hybrid,"paper, trained on KITTI (DPT Tab. 3)",0.959,0.062,2.573,0.092
"Ours (final, 3 seeds), zero-shot raw",measured,0.7682 ± 0.014,0.1511 ± 0.0012,3.34 ± 0.026,0.1883 ± 0.0017
"Ours (final, 3 seeds), zero-shot median-scaled",measured,0.8951 ± 0.0046,0.1118 ± 0.0035,4.096 ± 0.076,0.1438 ± 0.0032


SUN RGB-D test, NYUv2 model


,source,rmse,d1,abs_rel,rmse_log
model,,,,,
DINOv2 ViT-S/14 lin. 1,"paper, trained on ~24k NYU frames",0.477,—,—,—
DINOv2 ViT-S/14 lin. 4,"paper, trained on ~24k NYU frames",0.431,—,—,—
DINOv2 ViT-S/14 DPT,"paper, trained on ~24k NYU frames",0.409,—,—,—
"Ours (final, 3 seeds), raw",measured,0.6075 ± 0.0015,0.7531 ± 0.0012,0.2122 ± 0.0011,0.2437 ± 0.00042
"Ours (final, 3 seeds), median-scaled",measured,0.5149 ± 0.0019,0.8412 ± 0.00063,0.1662 ± 0.00086,0.207 ± 0.0005


Segmentation on the mapped class subsets


,classes,mIoU,pixAcc
model,,,
cityscapes -> ade20k,15,59 ± 0.73,86.86 ± 0.3
cityscapes -> voc2012,7,70.02 ± 1.8,96.4 ± 0.36
cityscapes -> pcontext,14,56.58 ± 1.3,79.81 ± 1.3
nyu -> ade20k,32,49.15 ± 0.23,76.96 ± 0.026
nyu -> voc2012,6,53.19 ± 0.26,93.73 ± 0.19
nyu -> pcontext,18,40.32 ± 0.58,68.71 ± 0.87


## Targets
Does the mean, and does every seed, beat each reference number? Single pass, except the multi-scale target vs DINOv2, which is TTA on both sides.
- **Cityscapes AbsRel / RMSElog / δ1:** a global scale bias. Predictions are ≈ 6 % too far (log bias +0.059), from training on 448×896 crops and testing on the full frame; the TTA rows above show the effect of matching the test geometry to training.
- **NYUv2 vs DINOv2 / DPT:** DINOv2 trains its depth head on ≈ 24k frames (frozen backbone), DPT-Hybrid is depth-pretrained on ≈ 1.4 M images (MIX 6) and fine-tuned on NYUv2; I train on 795 images.
- **SwinMTL's mIoU is compared with both metrics on both datasets:** the standard dataset-level mIoU (the comparable one) and SwinMTL's per-image mIoU. The per-image rows show where I win (NYU) **and** where I lose (Cityscapes). SwinMTL's Cityscapes number is not reproducible from their code.
- **vs DINOv2 (frozen):** my fine-tuned model against DINOv2's frozen-feature probes, not against a fine-tuned DINOv2.

In [10]:
TARGETS_TABLE = FINAL_TARGETS   # defined once in repro_common.py
rows = []
for label, ds, key, metric, paper, lower in TARGETS_TABLE:
    v = vals(ds, key, metric)
    if len(v) == 0:
        continue   # e.g. multi-scale without TTA
    beats = (v < paper) if lower else (v > paper)
    rows.append(dict(target=label, reference=paper, ours_mean=v.mean(), ours_std=v.std(ddof=1),
                     beats=bool((v.mean() < paper) if lower else (v.mean() > paper)), seeds_beating=f"{int(beats.sum())}/{len(v)}"))
targets = pd.DataFrame(rows).set_index("target")
targets.to_csv(OUT / "targets.csv")
display(targets.style.format(precision=4).apply(
    lambda r: ["background-color: rgba(0,180,0,.2)" if r["beats"] else "background-color: rgba(220,0,0,.15)"] * len(r), axis=1))

,reference,ours_mean,ours_std,beats,seeds_beating
target,,,,,
CS AbsRel vs SwinMTL,0.0890,0.1082,0.0021,False,0/3
CS SqRel vs SwinMTL,1.0510,0.8228,0.0027,True,3/3
CS RMSE (512) vs SwinMTL,5.4810,5.0244,0.0192,True,3/3
CS RMSElog vs SwinMTL,0.1390,0.1461,0.0009,False,0/3
CS δ1 vs SwinMTL,0.9210,0.9140,0.0003,False,0/3
CS RMSE (full) vs SwinMTL,6.3520,5.0591,0.0189,True,3/3
CS mIoU (standard) vs SwinMTL,76.4100,77.7365,0.1863,True,3/3
CS mIoU (SwinMTL per-image) vs SwinMTL,76.4100,65.2273,0.0170,False,0/3
"CS mIoU vs DINOv2 (frozen, lin.)",66.6000,77.7365,0.1863,True,3/3


## Reproduction check
Each reproduced evaluation is compared with the original one, saved by the development notebooks in `results/` (skipped if `results/` is not present).
- The table gives, per run, the number of metrics compared and the largest relative difference.
- Same weights and code give the same numbers. On another GPU, floating-point differences of about 1e-4 relative are expected.

In [11]:
rows = []
for dataset, runs in FINAL.items():
    for seed, (run, _) in runs.items():
        for kind, new, suffix in [("eval", EVAL.get((dataset, seed)), ""), ("zero-shot", ZS.get((dataset, seed)), "_zeroshot")]:
            if new is None:
                continue
            c = compare_with_original(new, load_json(original_eval_path(run, suffix)))
            rows.append(dict(run=f"{dataset} s{seed}", kind=kind, **(c or dict(n_metrics=0, max_rel_diff=None, worst="original not found"))))
check = pd.DataFrame(rows)
check.to_csv(OUT / "reproduction_check.csv", index=False)
display(check)
if check["max_rel_diff"].notna().any():
    print(f"largest relative difference over all runs: {check['max_rel_diff'].max():.2e}")

,run,kind,n_metrics,max_rel_diff,worst,new,original
0,cityscapes s42,eval,46,0.0,cs_depth_512x1024.d1,0.913638,0.913638
1,cityscapes s42,zero-shot,27,0.0,kitti_raw.d1,0.768524,0.768524
2,cityscapes s43,eval,46,0.0,cs_depth_512x1024.d1,0.914270,0.914270
3,cityscapes s43,zero-shot,27,0.0,kitti_raw.d1,0.754169,0.754169
4,cityscapes s44,eval,46,0.0,cs_depth_512x1024.d1,0.914166,0.914166
5,cityscapes s44,zero-shot,27,0.0,kitti_raw.d1,0.781805,0.781805
6,nyu s42,eval,35,0.0,nyu_depth.d1,0.863679,0.863679
7,nyu s42,zero-shot,27,0.0,sun_raw.d1,0.754406,0.754406
8,nyu s43,eval,35,0.0,nyu_depth.d1,0.866331,0.866331
9,nyu s43,zero-shot,27,0.0,sun_raw.d1,0.752440,0.752440


largest relative difference over all runs: 0.00e+00
